# 05 — Home/Office reliability validation

Frozen 27 HOME / 16 OFFICE is a parity checkpoint, not a target.

This notebook keeps CP1 stays, the frozen Beijing semantic policy, and complete-link 200 m. It re-evaluates the semantic layer with multiple rankers and reliability tests:

- candidate coverage
- cross-method agreement
- split-half test–retest reliability
- held-out predictive persistence
- stay-dropout robustness
- +12 h schedule-sensitivity stress

No metric below is called semantic accuracy because GeoLife has no HOME/OFFICE ground truth.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_05 = VOLUME_ROOT / "cache" / "05_home_office_reliability_validation"
CACHE_05.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def ensure_repo():
    module_path = REPO_DIR / "analysis" / "05_home_office_reliability.py"
    if (REPO_DIR / ".git").exists():
        current = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
            text=True,
        ).strip()
        if current != REPO_BRANCH:
            run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
        refresh = os.environ.get("GEOLIFE_REFRESH_REPO", "0") == "1"
        if refresh or not module_path.exists():
            run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"])
    else:
        run([
            "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
            REPO_URL, str(REPO_DIR),
        ])

ensure_repo()
for candidate in (REPO_DIR, REPO_DIR / "src"):
    value = str(candidate)
    if value not in sys.path:
        sys.path.insert(0, value)

try:
    import numpy as np
    import pandas as pd
    from geolife.model import HomeOfficeConfig, build_semantic_locations
except ModuleNotFoundError:
    run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR)
    import numpy as np
    import pandas as pd
    from geolife.model import HomeOfficeConfig, build_semantic_locations

module_path = REPO_DIR / "analysis" / "05_home_office_reliability.py"
spec = importlib.util.spec_from_file_location("home_office_reliability_05", module_path)
reliability = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = reliability
spec.loader.exec_module(reliability)

print("Ready")
print("Cache:", CACHE_05)

## 1. Frozen input and semantic cohort

The expensive raw-GPS path is not rerun. We load the private CP1 stay cache and rebuild only the frozen semantic locations.

In [ ]:
def find_stay_cache():
    candidates = [
        VOLUME_ROOT / "cache" / "03a_user_behavior_deep_dive" / "stays_baseline_v1.pkl",
        VOLUME_ROOT / "artifacts" / "03a" / "stays_baseline_v1.pkl",
        REPO_DIR / "artifacts" / "03a" / "stays_baseline_v1.pkl",
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob("**/stays_baseline_v1.pkl"))
    if matches:
        return matches[0]
    raise FileNotFoundError("Cannot find stays_baseline_v1.pkl")

STAY_CACHE = find_stay_cache()
stays = pd.read_pickle(STAY_CACHE)

cfg = HomeOfficeConfig(
    clustering_method="complete_link",
    location_max_diameter_m=200.0,
)

semantic_stays, locations = build_semantic_locations(stays, config=cfg)
recurring = locations.loc[locations["stay_count"] >= 2]

print("CP1 stays:", len(stays))
print("CP1 stay users:", stays["user_id"].nunique())
print("Semantic users:", semantic_stays["user_id"].nunique())
print("Semantic locations:", len(locations))
print("Recurring locations:", len(recurring))
print("Recurring-anchor users:", recurring["user_id"].nunique())

## 2. Baseline parity

Stop if the frozen production comparator no longer reproduces 27 HOME / 16 OFFICE.

In [ ]:
parity = reliability.baseline_emission_parity(stays, config=cfg)
display(parity)

parity_map = dict(zip(parity["label"], parity["emitted_users"], strict=True))
assert parity_map.get("HOME") == 27
assert parity_map.get("OFFICE") == 16

display(pd.DataFrame([reliability.synthetic_self_check()]))

## 3. Candidate coverage and cross-method convergence

The fixed-window rows here are top supported candidates before final share/margin emission gates, so coverage is allowed to exceed 27/16.

In [ ]:
assignments = reliability.infer_assignments(semantic_stays, config=cfg)

coverage = (
    assignments.groupby(["method", "label"], as_index=False)
    .agg(
        candidate_users=("user_id", "nunique"),
        emitted_users=("emitted", "sum"),
        median_support_days=("support_days", "median"),
        median_score=("score", "median"),
    )
)

cross_method = reliability.cross_method_agreement(assignments)

display(coverage)
display(cross_method)

assignments.to_pickle(CACHE_05 / "assignments_private.pkl")
coverage.to_csv(CACHE_05 / "candidate_coverage.csv", index=False)
cross_method.to_csv(CACHE_05 / "cross_method_agreement.csv", index=False)

## 4. Split-half test–retest reliability

Compare first-vs-second observed dates and odd-vs-even ISO weeks. Location IDs remain frozen so this isolates semantic assignment stability.

In [ ]:
split_summary, split_details = reliability.split_half_reliability(
    semantic_stays,
    config=cfg,
)

display(split_summary)

split_details.to_pickle(CACHE_05 / "split_half_details_private.pkl")
split_summary.to_csv(CACHE_05 / "split_half_summary.csv", index=False)

## 5. Held-out predictive persistence

Infer on the first 60% of each user's observed dates, then evaluate the frozen candidate on the last 40%.

HOME is evaluated by held-out dwell-share rank. OFFICE is evaluated by held-out weekday-visit-day-share rank.

In [ ]:
holdout_summary, holdout_details = reliability.heldout_predictive_validity(
    semantic_stays,
    config=cfg,
)

display(holdout_summary)

holdout_details.to_pickle(CACHE_05 / "holdout_details_private.pkl")
holdout_summary.to_csv(CACHE_05 / "holdout_summary.csv", index=False)

## 6. Missing-data robustness

Randomly drop 10%, 20%, and 30% of stay events with deterministic seeds and measure candidate retention.

In [ ]:
dropout_summary, dropout_details = reliability.dropout_robustness(
    semantic_stays,
    config=cfg,
)

display(dropout_summary)

dropout_details.to_pickle(CACHE_05 / "dropout_details_private.pkl")
dropout_summary.to_csv(CACHE_05 / "dropout_summary.csv", index=False)

## 7. +12 h schedule-sensitivity stress

Physical locations stay unchanged. The recurrence method is a schedule-light control; fixed-window and HoWDe-style methods are expected to be clock-sensitive by design.

In [ ]:
shift_summary, shift_details = reliability.time_shift_stress(
    semantic_stays,
    hours=12,
    config=cfg,
)

display(shift_summary)

shift_details.to_pickle(CACHE_05 / "time_shift_details_private.pkl")
shift_summary.to_csv(CACHE_05 / "time_shift_summary.csv", index=False)

## 8. Interpretation

Do not pick a method only because it emits more users.

- high coverage + low reliability: over-eager
- low coverage + high reliability: conservative but stable
- high split-half reliability + held-out persistence: stronger behavioral evidence
- cross-method disagreement: method-dependent semantics
- dropout sensitivity: observation-support dependence
- time-shift sensitivity: schedule dependence

These are reliability signals, not ground-truth accuracy.

In [ ]:
tables = {
    "candidate_coverage": coverage,
    "cross_method": cross_method,
    "split_half": split_summary,
    "holdout": holdout_summary,
    "dropout": dropout_summary,
    "time_shift": shift_summary,
}

for name, frame in tables.items():
    path = CACHE_05 / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(path)

print("Private user-level details:", CACHE_05)